# Phase 1 Baseline — Regex Seeds + Universal Defaults

**Goal**: end-to-end submission.csv working. No models, CPU only.

**Expected val Macro F1 ≈ 0.13** (calibrated locally).

This notebook validates 3 critical things before we add anything heavy:
1. submission.csv format is accepted by Kaggle (`test_001` ids, `;` separator)
2. citations we emit `exact match` the corpus vocabulary
3. local val F1 ↔ public LB F1 are consistent

Run all cells top-to-bottom. Final cell writes `/kaggle/working/submission.csv`.

In [ ]:
# Cell 1 — paths & libs (no GPU, no pip-install needed)
import re, csv, sys
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd

DATA = Path('/kaggle/input/llm-agentic-legal-information-retrieval')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv', 'court_considerations.csv']:
    f = DATA / p
    assert f.exists(), f'missing: {f}'
    print(f'  ok: {p}  ({f.stat().st_size/1e6:.1f} MB)')


In [ ]:
# Cell 2 — normalization module (inline)

MULTILANG_ABBR = {
    'LAI':'IVG','LAA':'UVG','LAVS':'AHVG','LAMal':'KVG',
    'LPP':'BVG','LPGA':'ATSG','LACI':'AVIG',
    'CO':'OR','CC':'ZGB','CP':'StGB','CPP':'StPO','CPC':'ZPO',
    'LTF':'BGG','LDIP':'IPRG','LIFD':'DBG','LP':'SchKG',
    'LPM':'MSchG','LCD':'UWG','LDA':'URG','LCart':'KG',
    'LCR':'SVG','LStup':'BetmG','Cst.':'BV','Cst':'BV',
}

ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-zÄÖÜäöüß0-9\.\-]+)$'
)

def parse_article(cit):
    m = ART_PATTERN.match(cit.strip())
    return m.groupdict() if m else None

def normalize_whitespace(cit):
    s = re.sub(r'\s+', ' ', cit.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    s = re.sub(r'\bZiff\.(?=\d)', 'Ziff. ', s)
    return s

def expand_multilang(cit):
    out = [cit]
    m = parse_article(cit)
    if m and m['code'] in MULTILANG_ABBR:
        out.append(cit[:cit.rfind(m['code'])] + MULTILANG_ABBR[m['code']])
    return out

print('Building corpus index ...')
laws = pd.read_csv(DATA/'laws_de.csv')
laws_cits = set(laws['citation'].astype(str))
print(f'  laws citations: {len(laws_cits):,}')

court_cits = set()
with open(DATA/'court_considerations.csv', newline='', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
print(f'  court citations: {len(court_cits):,}')

corpus_cits = laws_cits | court_cits
print(f'  combined: {len(corpus_cits):,}')

parent_to_children = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    p = m.groupdict()
    parent = f"Art. {p['art']} {p['code']}"
    if c != parent:
        parent_to_children[parent].append(c)
print(f'  parents with children: {len(parent_to_children):,}')

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in corpus_cits: return [c], 'exact'
    for v in expand_multilang(c):
        if v != c and v in corpus_cits: return [v], 'multilang_abbr'
    if c in parent_to_children:
        return list(parent_to_children[c]), 'parent_to_children'
    p = parse_article(c)
    if p:
        parent = f"Art. {p['art']} {p['code']}"
        if parent != c and parent in corpus_cits:
            return [parent], 'child_to_parent'
    return [], 'no_match'

def granularity_filter(cits):
    cit_set = set(cits)
    out = []
    for c in cits:
        ch = parent_to_children.get(c)
        if ch and any(x in cit_set for x in ch): continue
        out.append(c)
    return out

def dedup(cits):
    seen = set(); out=[]
    for c in cits:
        if c in seen: continue
        seen.add(c); out.append(c)
    return out


In [ ]:
# Cell 3 — verify normalization on train gold (expected ≥ 97%)

def split_citations(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [c.strip() for c in s.split(';') if c.strip()]

train = pd.read_csv(DATA/'train.csv')
strategy = Counter(); total=0; resolved=0
for _, r in train.iterrows():
    for c in split_citations(r['gold_citations']):
        total += 1
        out, tag = resolve_against_corpus(c)
        strategy[tag] += 1
        if out: resolved += 1
print(f'TRAIN gold resolve rate: {resolved}/{total} = {100*resolved/total:.2f}%')
for k, v in strategy.most_common():
    print(f'  {k:<24} {v:>5} ({100*v/total:.2f}%)')
assert resolved/total >= 0.97, 'normalize fail: expected ≥97%'


In [ ]:
# Cell 4 — universal defaults + regex seed extraction

UNIVERSAL_DEFAULTS = ['Art. 100 Abs. 1 BGG']

RE_ART_Q = re.compile(
    r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
    r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
    r'\s+([A-Z][A-Za-zÄÖÜäöüß0-9\.\-]+)'
)
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def _resolve_each(cits):
    out=[]
    for c in cits:
        r,_ = resolve_against_corpus(c)
        out.extend(r)
    return out

def extract_seeds(query):
    seeds=[]
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, code = m.groups()
        # Try most-specific first, then drop modifiers progressively
        candidates = []
        if abs_ and lit:
            candidates.append(f'Art. {art} Abs. {abs_} lit. {lit} {code}')
            candidates.append(f'Art. {art} Abs. {abs_} {code}')
        elif abs_:
            candidates.append(f'Art. {art} Abs. {abs_} {code}')
        candidates.append(f'Art. {art} {code}')
        for cand in candidates:
            r,_ = resolve_against_corpus(cand)
            if r:
                seeds.extend(r); break
    for m in RE_BGE_Q.finditer(query):
        vol, book, page, e = m.groups()
        base = f'BGE {vol} {book} {page}'
        if e:
            r,_ = resolve_against_corpus(f'{base} E. {e}')
            seeds.extend(r)
        r,_ = resolve_against_corpus(base); seeds.extend(r)
    for m in RE_BGER_Q.finditer(query):
        case, e = m.groups()
        if e:
            r,_ = resolve_against_corpus(f'{case} E. {e}'); seeds.extend(r)
        r,_ = resolve_against_corpus(case); seeds.extend(r)
    return dedup(seeds)

def predict(query):
    seeds = extract_seeds(query)
    defaults = _resolve_each(UNIVERSAL_DEFAULTS)
    merged = dedup(seeds + defaults)
    return granularity_filter(merged)


In [ ]:
# Cell 5 — evaluate on val (expected Macro F1 ~0.13)

def per_query_f1(gold, pred):
    g, p = set(gold), set(pred)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp = len(g & p)
    if tp == 0: return 0.0
    pr = tp/len(p); rc = tp/len(g)
    return 2*pr*rc/(pr+rc)

def macro_f1(gold_dict, pred_dict):
    qids = set(gold_dict)|set(pred_dict)
    return sum(per_query_f1(gold_dict.get(q,[]), pred_dict.get(q,[])) for q in qids)/max(1,len(qids))

val = pd.read_csv(DATA/'val.csv')
gold_dict = {r['query_id']: split_citations(r['gold_citations']) for _, r in val.iterrows()}
pred_dict = {r['query_id']: predict(r['query']) for _, r in val.iterrows()}
f1 = macro_f1(gold_dict, pred_dict)
print(f'VAL Macro F1 = {f1:.4f}')
for qid in sorted(pred_dict):
    g, p = gold_dict[qid], pred_dict[qid]
    print(f'  {qid}: pred={len(p):>2}  gold={len(g):>2}  F1={per_query_f1(g,p):.3f}')


In [ ]:
# Cell 6 — generate submission.csv

test = pd.read_csv(DATA/'test.csv')
rows = []
for _, r in test.iterrows():
    cits = predict(r['query'])
    rows.append({'query_id': r['query_id'], 'predicted_citations': ';'.join(cits)})

sub = pd.DataFrame(rows)
out_path = WORK/'submission.csv'
sub.to_csv(out_path, index=False)
print(f'Wrote {out_path}  rows={len(sub)}')
print('\nfirst 5 rows:')
print(sub.head().to_string(index=False))
print('\ncitation counts per row:')
print(sub['predicted_citations'].str.count(';').add(1).describe())
